In [1]:
pip install langchain langchain-community langchain-chroma langchain-groq langchain-huggingface pypdf streamlit sentence-transformers chromadb python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.2/504.2 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 

In [2]:
import os
import streamlit as st
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import gradio as gr
import gradio as gr
from kaggle_secrets import UserSecretsClient
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


In [3]:
user_secrets = UserSecretsClient()
GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")

# ====================== CONFIG ======================
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2,
    groq_api_key=GROQ_API_KEY
)

vectorstore = None
retriever = None

# ====================== PROMPT ======================
template = """You are an expert cybersecurity researcher specializing in network attacks and intrusion detection.

Answer ONLY using the provided context from the uploaded research papers.
If you don't know, say: "I don't have enough information from the uploaded documents."

Context:
{context}

Question: {question}

Answer professionally with bullet points for models, features, metrics. Cite source when possible."""
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(f"Source: {doc.metadata.get('source', 'PDF')}\n{doc.page_content}" for doc in docs)

# ====================== BUILD ======================
def build_knowledge_base(pdf_files):
    global vectorstore, retriever
    if not pdf_files:
        return "Please upload at least one PDF."
    all_splits = []
    for pdf in pdf_files:
        loader = PyPDFLoader(pdf.name)
        pages = loader.load()
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        splits = text_splitter.split_documents(pages)
        all_splits.extend(splits)

    if not all_splits:
        return "No text extracted."

    vectorstore = Chroma.from_documents(documents=all_splits, embedding=embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 6})
    return f"✅ Success! {len(all_splits)} chunks loaded.\nYour paper is ready!"

# ====================== CHAT — FIXED WITH .copy() ======================
def chat(message, history):
    if history is None:
        history = []
    else:
        history = list(history)          # ←←← This line fixes the tuple error

    if retriever is None:
        history.append([message, "Please upload PDF and click Build Knowledge Base first!"])
        return history

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    response = rag_chain.invoke(message)
    
    history.append([message, response])
    return history

# ====================== CLEAR ======================
def clear_chat():
    return []

# ====================== INTERFACE ======================
with gr.Blocks(title="Cyber RAG Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🔒 RAG Cybersecurity Knowledge Assistant\n**Your Network_Attack_CICDDos2019 paper is loaded!**")

    with gr.Row():
        with gr.Column(scale=1):
            pdf_upload = gr.File(label="Upload PDF(s)", file_count="multiple", file_types=[".pdf"])
            build_btn = gr.Button("🚀 Build Knowledge Base", variant="primary", size="large")
            status = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=600, show_copy_button=True)
            msg = gr.Textbox(label="Ask anything about your research", placeholder="Summarize the feature engineering...")
            clear_btn = gr.Button("Clear Chat")

    build_btn.click(build_knowledge_base, inputs=pdf_upload, outputs=status)
    msg.submit(chat, inputs=[msg, chatbot], outputs=chatbot)
    clear_btn.click(clear_chat, outputs=chatbot)

demo.launch(share=True, server_name="0.0.0.0", server_port=7861, debug=False)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_17/1007819952.py:80: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Cyber RAG Assistant", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_17/1007819952.py:90: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=600, show_copy_button=True)
/tmp/ipykernel_17/1007819952.py:90: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(height=600, show_copy_button=True)
/tmp/ipykernel_17/1007819952.py:90: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbo

* Running on local URL:  http://0.0.0.0:7861
* Running on public URL: https://798e1615ee0f91e45e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
